In [2]:
# =========================
# Import Libraries
# =========================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib

# =========================
# Load Dataset
# =========================
dataset = pd.read_csv('dataset/kidney.csv')

# =========================
# Preprocessing
# =========================
dataset.replace('?', np.nan, inplace=True)
dataset = dataset.apply(lambda x: x.fillna(x.value_counts().index[0]))

# Encode categorical
le = LabelEncoder()
for col in dataset.columns:
    if dataset[col].dtype == 'object':
        dataset[col] = le.fit_transform(dataset[col])

# Split
X = dataset.drop('classification', axis=1)
y = dataset['classification']

# Save column names ✅
feature_cols = X.columns.tolist()

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.33, random_state=0
)

# =========================
# Train Models
# =========================

# KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# SVM
svc = SVC(kernel='rbf', probability=True)
svc.fit(X_train, y_train)

# Random Forest (BEST usually)
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)

# 👉 Choose best (you can improve later)
model = rf

# =========================
# Save Model
# =========================
joblib.dump(model, 'kidney_model.pkl')
joblib.dump(scaler, 'kidney_scaler.pkl')
joblib.dump(feature_cols, 'kidney_features.pkl')  # ✅ FIXED

print("✅ Kidney model saved correctly!")

# =========================
# Prediction Function
# =========================
def predict_kidney(new_data):
    model = joblib.load('kidney_model.pkl')
    scaler = joblib.load('kidney_scaler.pkl')
    features = joblib.load('kidney_features.pkl')

    df = pd.DataFrame([new_data])

    # Add missing columns with default value
    for col in features:
        if col not in df:
            df[col] = 0

    # Ensure correct column order
    df = df[features]

    # Scale
    df = scaler.transform(df)

    prob = model.predict_proba(df)[0][1] * 100

    if prob >= 70:
        risk = "HIGH RISK"
    elif prob >= 30:
        risk = "MEDIUM RISK"
    else:
        risk = "LOW RISK"

    print(f"\n🎯 Kidney Disease Probability: {prob:.2f}% | Risk: {risk}")

✅ Kidney model saved correctly!
